# Testing All Functionalities of the Environment

In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym

# Adjust these imports to your project layout
import env_config
from battery_env import BatteryEnv 

# Instantiate env
env = BatteryEnv(
    env_config,
    publish_hour=13,
    episode_days=7,              
    randomize_init_soc=False,  
    seed=123,
)

obs, info = env.reset(seed=123)
print("obs shape:", np.array(obs).shape)
print("info:", info)
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)


obs shape: (54,)
info: {'start_day': '2021-01-12T00:00:00.000000000', 'publish_hour': 13, 'episode_days': 7, 'init_soc': 0.5}
action_space: MultiDiscrete([11 11])
observation_space: Box([ 0.0000e+00 -3.0000e+02  0.0000e+00  1.0000e+00 -1.8635e-01  0.0000e+00
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02], [1.0000e+00 2.5000e+03 1.0000e+03 4.8000e+01 1.8635e-01 1.0000e+00
 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03
 2.5000e+03 2.5000e+0

## 1.1. Quick Environment Structure Check

In [2]:
import numpy as np
import pandas as pd
import gymnasium as gym

# Adjust these imports to your project layout
import env_config
from battery_env import BatteryEnv 

# Instantiate env
env = BatteryEnv(
    env_config,
    publish_hour=13,
    episode_days=7,              
    randomize_init_soc=False,  
    seed=123,
)

obs, info = env.reset(seed=123)

action = env.action_space.sample()
obs, r, term, trunc, info = env.step(action)

print("Obs shape: ", obs.shape)
print("Obs length: ", len(obs))
print("Expected length: ", 54)
assert len(obs) == 54, "Obs length mismatch; exepcted 54 = 6  main + 48 curve"

print("Obs space low/high lengths:", len(env.observation_space.low), len(env.observation_space.high))
assert len(env.observation_space.low) == 54
assert len(env.observation_space.high) == 54

print("valid days:", len(env.valid_days))
print("first valid days: ", env.valid_days[0], "last:", env.valid_days[-1])

assert (obs[6:] != 0).any() == info["da_available"]


Obs shape:  (54,)
Obs length:  54
Expected length:  54
Obs space low/high lengths: 54 54
valid days: 728
first valid days:  2021-01-01T00:00:00.000000000 last: 2022-12-31T00:00:00.000000000


## 1.2. Pick one Day and Print its 48 slots (table)

In [3]:
#pick the current day the env started on
day = env.current_day if hasattr(env, "current_day") else env.current_delivery_day
idxs = env.day_indices[day]

df_day = env.df.loc[idxs, ["trade_ts", "delivery_date", "tau", "price_gbp_mwh", "carbon_gco2_kwh"]].copy()
df_day = df_day.sort_values("trade_ts").reset_index(drop=True)

display(df_day.head(10))
display(df_day.tail(10))

# Hard assertions
assert len(df_day) == 48, f"Day has {len(df_day)} rows, exepcted 48"
assert set(df_day["tau"].tolist()) == set(range(1,49)), "tau is not exactly 1,48"

deltas = df_day["trade_ts"].diff().dropna()
assert (deltas == pd.Timedelta(minutes=30)).all(), "timestamps  are not exaclty 30 minutes "

print ("Day check passed:", day)

,trade_ts,delivery_date,tau,price_gbp_mwh,carbon_gco2_kwh
0,2021-01-12 00:00:00,2021-01-13,1,59.5,149.0
1,2021-01-12 00:30:00,2021-01-13,2,72.5,149.0
2,2021-01-12 01:00:00,2021-01-13,3,65.4,149.0
3,2021-01-12 01:30:00,2021-01-13,4,65.0,140.0
4,2021-01-12 02:00:00,2021-01-13,5,65.1,131.0
5,2021-01-12 02:30:00,2021-01-13,6,62.3,132.0
6,2021-01-12 03:00:00,2021-01-13,7,65.4,141.0
7,2021-01-12 03:30:00,2021-01-13,8,62.0,138.0
8,2021-01-12 04:00:00,2021-01-13,9,64.0,137.0
9,2021-01-12 04:30:00,2021-01-13,10,60.0,143.0


,trade_ts,delivery_date,tau,price_gbp_mwh,carbon_gco2_kwh
38,2021-01-12 19:00:00,2021-01-13,39,199.0,284.0
39,2021-01-12 19:30:00,2021-01-13,40,96.6,286.0
40,2021-01-12 20:00:00,2021-01-13,41,102.0,292.0
41,2021-01-12 20:30:00,2021-01-13,42,74.8,293.0
42,2021-01-12 21:00:00,2021-01-13,43,80.0,295.0
43,2021-01-12 21:30:00,2021-01-13,44,66.5,296.0
44,2021-01-12 22:00:00,2021-01-13,45,65.4,296.0
45,2021-01-12 22:30:00,2021-01-13,46,60.0,287.0
46,2021-01-12 23:00:00,2021-01-13,47,65.1,279.0
47,2021-01-12 23:30:00,2021-01-13,48,64.8,277.0


Day check passed: 2021-01-12T00:00:00.000000000


## 1.3. Validate all valid days match the rules

In [4]:
bad = []
for d in env.valid_days:
    idxs = env.day_indices[d]
    df_d = env.df.loc[idxs, ["trade_ts", "tau"]].sort_values("trade_ts")
    if len(df_d) != 48:
        bad.append((d, "len", len(df_d)))
        continue
    if set(df_d["tau"].tolist()) != set(range(1,49)):
        bad.append(d, "tau_set", sorted(set(df_d["tau"].tolist())[:10]))
        continue
    deltas = df_d["trade_ts"].diff().dropna()
    if not(deltas == pd.Timedelta(minutes=30)).all():
        bad.append(d, "cadence", deltas.value_counts().head(3).to_dict())
        continue

print('Bad days count:', len(bad))
if bad:
    print("First 5 bad:",bad[:5])
assert len(bad) == 0, "Some valid days violate 48/tau/cadence rules."
print("All valids_days passed integrity checks")



Bad days count: 0
All valids_days passed integrity checks


In [5]:
if max(env.tau) == 48:
    print("Tau = 48")
else:
    print("Tay XX<=48")

if min(env.tau) == 1:
    print("Tau = 1")
else:
    print("Tau XXX>0")

if min(env.ci) >= 0:
    print("CI Min > 0")
else:
    print("CI Min XXX>0")

if max(env.ci) <= 1000:
    print("CI Max < 1000")
else:
    print("CI Max XXX<1000")

Tau = 48
Tau = 1
CI Min > 0
CI Max < 1000


In [6]:
import numpy as np
import pandas as pd
import env_config
from battery_env import BatteryEnv

def test_minus1_means_no_plan(env, N=48, seed=123):
    obs, info = env.reset(seed=seed)

    # Force: no DA commitments for the whole day
    env.today_plan[:] = -1
    env.tomorrow_plan[:] = -1

    rows = []
    fails = 0

    for k in range(N):
        a = env.action_space.sample()
        agent_dispatch = int(a[0])

        obs, r, term, trunc, info = env.step(a)

        # You should add this to info in env.step() for clean testing:
        # info["dispatch_idx_exec"] = dispatch_idx_eff
        exec_dispatch = info.get("dispatch_idx_exec", None)

        # If you didn't add it, we can infer from P_req_MW (less robust if protection clips),
        # so strongly recommend adding dispatch_idx_exec to info.
        if exec_dispatch is None:
            raise RuntimeError("Add info['dispatch_idx_exec'] in env.step() to run this test cleanly.")

        ok = (exec_dispatch == agent_dispatch)
        fails += (not ok)

        rows.append({
            "k": k,
            "tau": info["tau"],
            "agent_dispatch": agent_dispatch,
            "exec_dispatch": exec_dispatch,
            "ok": ok,
            "P_req_MW": info["P_req_MW"],
            "P_app_MW": info["P_applied_MW"],
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("fails:", fails, "out of", len(df))
    display(df[df["ok"] == False].head(10))
    return df

env = BatteryEnv(env_config)
df = test_minus1_means_no_plan(env)

fails: 0 out of 48


,k,tau,agent_dispatch,exec_dispatch,ok,P_req_MW,P_app_MW


In [7]:
def get_idle_index(env):
    return int(np.argmin(np.abs(env.power_levels)))

env = BatteryEnv(env_config)
idle_idx = get_idle_index(env)
print("idle_idx:", idle_idx, "power:", env.power_levels[idle_idx])

idle_idx: 5 power: 0.0


In [8]:
import numpy as np
import pandas as pd
import env_config
from battery_env import BatteryEnv

def test_plan_idle_overrides_agent(env, N=20, seed=123):
    obs, info = env.reset(seed=seed)

    idle_idx = int(np.argmin(np.abs(env.power_levels)))

    # Force: every slot has a planned idle commitment
    env.today_plan[:] = idle_idx

    rows = []
    fails = 0

    for k in range(N):
        a = env.action_space.sample()
        agent_dispatch = int(a[0])

        obs, r, term, trunc, info = env.step(a)
        exec_dispatch = info.get("dispatch_idx_exec", None)

        if exec_dispatch is None:
            raise RuntimeError("Add info['dispatch_idx_exec'] in env.step()")

        ok = (exec_dispatch == idle_idx)  # should ALWAYS be idle
        fails += (not ok)

        rows.append({
            "k": k,
            "tau": info["tau"],
            "agent_dispatch": agent_dispatch,
            "exec_dispatch": exec_dispatch,
            "idle_idx": idle_idx,
            "ok": ok,
            "P_req_MW": info["P_req_MW"],
            "P_app_MW": info["P_applied_MW"],
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("fails:", fails, "out of", len(df))
    display(df[df["ok"] == False].head(10))
    return df
print(df)

env = BatteryEnv(env_config)
df = test_plan_idle_overrides_agent(env)

     k  tau  agent_dispatch  exec_dispatch    ok  P_req_MW      P_app_MW
0    0    1               7              7  True   0.07454  7.454000e-02
1    1    2               6              6  True   0.03727  3.727000e-02
2    2    3               6              6  True   0.03727  3.727000e-02
3    3    4               4              4  True  -0.03727 -3.727000e-02
4    4    5               6              6  True   0.03727  3.727000e-02
5    5    6               9              9  True   0.14908  1.338324e-01
6    6    7               3              3  True  -0.07454 -7.454000e-02
7    7    8               7              7  True   0.07454  7.352840e-02
8    8    9               7              7  True   0.07454  2.101430e-16
9    9   10               2              2  True  -0.11181 -1.118100e-01
10  10   11               6              6  True   0.03727  3.727000e-02
11  11   12               2              2  True  -0.11181 -1.118100e-01
12  12   13               7              7  True   

,k,tau,agent_dispatch,exec_dispatch,idle_idx,ok,P_req_MW,P_app_MW


# 2. Timeline Correctness

## 2.1. Is the step through time done correctly

This test verifies that one environment step corresponds to exactly one 30-minute
interval in the dataset. We check that timestamps advance correctly, tau increments
from 1 to 48 without skips, and the environment remains within the same delivery day
until a full day (48 steps) is completed. This ensures that the temporal structure of
the environment is consistent with the physical market timeline.

In [9]:
import pandas as pd
import numpy as np


def run_time_step_sanity(env, n_steps=60, seed=123):
    obs, info = env.reset(seed=seed)
    print("Reset info:", info)
    print("Start trade day (env.current_day):", env.current_day, "| slot0:", env.slot0, "| days_done:", env.days_done)
    print()

    rows = []
    for k in range(n_steps):
        obs_before = obs.copy()                 # <- state s_t
        action = env.action_space.sample()
        obs, r, term, trunc, inf = env.step(action)  # obs is now s_{t+1}

        rows.append({
            "k": k,
            "trade_ts": inf["trade_ts"],
            "trade_day": pd.to_datetime(inf["trade_ts"]).floor("D"),
            "tau": inf["tau"],
            "da_available": inf["da_available"],
            "tomorrow_curve_nonzero": bool(np.any(obs_before[6:] != 0.0)),
        })

        if term or trunc:
            break

    df_print = pd.DataFrame(rows)

    # 1) 30-min cadence check on trade_ts
    ts = pd.to_datetime(df_print["trade_ts"])
    df_print["dt_min"] = ts.diff().dt.total_seconds() / 60.0

    # 2) expected DA gate time for the first trade day we are in
    # DA should become available at trade_day 13:00 (or your env.publish_hour)
    trade_day0 = pd.to_datetime(df_print["trade_day"].iloc[0]).floor("D")
    publish_ts = trade_day0 + pd.Timedelta(hours=env.publish_hour)
    df_print["publish_ts_expected"] = str(publish_ts)

    return df_print

df_t = run_time_step_sanity(env, n_steps=60, seed=123)
df_t.head(35)

Reset info: {'start_day': '2022-02-19T00:00:00.000000000', 'publish_hour': 13, 'episode_days': 30, 'init_soc': 0.5729407452992573}
Start trade day (env.current_day): 2022-02-19T00:00:00.000000000 | slot0: 0 | days_done: 0



,k,trade_ts,trade_day,tau,da_available,tomorrow_curve_nonzero,dt_min,publish_ts_expected
0,0,2022-02-19 00:00:00,2022-02-19,1,False,False,NaN,2022-02-19 13:00:00
1,1,2022-02-19 00:30:00,2022-02-19,2,False,False,30.0,2022-02-19 13:00:00
2,2,2022-02-19 01:00:00,2022-02-19,3,False,False,30.0,2022-02-19 13:00:00
3,3,2022-02-19 01:30:00,2022-02-19,4,False,False,30.0,2022-02-19 13:00:00
4,4,2022-02-19 02:00:00,2022-02-19,5,False,False,30.0,2022-02-19 13:00:00
5,5,2022-02-19 02:30:00,2022-02-19,6,False,False,30.0,2022-02-19 13:00:00
6,6,2022-02-19 03:00:00,2022-02-19,7,False,False,30.0,2022-02-19 13:00:00
7,7,2022-02-19 03:30:00,2022-02-19,8,False,False,30.0,2022-02-19 13:00:00
8,8,2022-02-19 04:00:00,2022-02-19,9,False,False,30.0,2022-02-19 13:00:00
9,9,2022-02-19 04:30:00,2022-02-19,10,False,False,30.0,2022-02-19 13:00:00


## 2.2. SoC Physics Sanity 

In this test, I run the environment for a large number of steps using randomly sampled actions to stress-test the battery dynamics. At each timestep, I verify that the state of charge (SoC) remains within the physical operating limits defined by SoC_min and SoC_max, allowing for a small numerical tolerance.

This check ensures that:
	•	the SoC protection logic correctly prevents over-charging and over-discharging,
	•	the current and power clamping in the battery model are consistent with the Coulomb-counting update,
	•	no numerical instability (e.g. NaNs or drift outside bounds) appears over long simulations.

If a violation is detected, the test logs the timestep, action taken, requested and applied power, reward, and timestamp, and stops immediately to simplify debugging.

In [10]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

eps = 1e-6
N_steps = 5000

obs, info = env.reset(seed=123)
rows = []
fails = []

for t in range(N_steps):
    soc_before = float(env.soc)

    a = env.action_space.sample()          # array([dispatch_idx, plan_idx])
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    delta = soc_after - soc_before

    P_req = float(env.power_levels[dispatch_idx])
    P_app = float(info.get("P_applied_MW", np.nan))

    # --- sign / direction sanity ---
    ok = True

    # Charge (negative power) => SoC should increase (or stay flat if blocked at max)
    if P_app < -eps and not (soc_after >= soc_before - 1e-8):
        ok = False

    # Discharge (positive power) => SoC should decrease (or stay flat if blocked at min)
    if P_app > eps and not (soc_after <= soc_before + 1e-8):
        ok = False

    # Near-zero power => SoC should barely move
    if abs(P_app) <= eps and not (abs(delta) <= 1e-6):
        ok = False

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": P_req,
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)

    if not ok:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
print("ran steps:", len(df))
print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))
else:
    display(df.head(20))

ran steps: 1440
num fails: 0


,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,3,1,-0.07454,-0.074540,0.572941,0.668377,0.095436,True,-1613.790928,2022-02-19 00:00:00,1,False
1,1,6,0,0.03727,0.037270,0.668377,0.620144,-0.048233,True,823.890583,2022-02-19 00:30:00,2,False
2,2,10,6,0.18635,0.186350,0.620144,0.374593,-0.245551,True,4541.349500,2022-02-19 01:00:00,3,False
3,3,4,10,-0.03727,-0.037270,0.374593,0.422605,0.048012,True,-890.380260,2022-02-19 01:30:00,4,False
4,4,4,10,-0.03727,-0.037270,0.422605,0.470610,0.048005,True,-907.479736,2022-02-19 02:00:00,5,False
5,5,2,0,-0.11181,-0.111810,0.470610,0.613653,0.143042,True,-2821.525318,2022-02-19 02:30:00,6,False
6,6,10,7,0.18635,0.186350,0.613653,0.367862,-0.245790,True,4788.114170,2022-02-19 03:00:00,7,False
7,7,2,0,-0.11181,-0.111810,0.367862,0.510965,0.143102,True,-2870.889433,2022-02-19 03:30:00,8,False
8,8,9,5,0.14908,0.149080,0.510965,0.313849,-0.197116,True,3896.205627,2022-02-19 04:00:00,9,False
9,9,10,4,0.18635,0.086433,0.313849,0.200000,-0.113849,True,2219.160982,2022-02-19 04:30:00,10,False


### 2.2.1. Charging Battery

In [11]:
charge_idx = 0                 # most negative power
discharge_idx = env.n_power_levels - 1  # most positive power

obs, info = env.reset(seed=123)

soc_before = env.soc

action = np.array([charge_idx, 0])  # dispatch charge, ignore planning
obs, r, term, trunc, info = env.step(action)

soc_after = env.soc

print("SoC before:", soc_before)
print("SoC after :", soc_after)
print("ΔSoC      :", soc_after - soc_before)
print("Applied MW:", info["P_applied_MW"])

SoC before: 0.5729407452992573
SoC after : 0.8092611446495056
ΔSoC      : 0.2363203993502483
Applied MW: -0.18635


### 2.2.2. Discharging battery

In [12]:
charge_idx = 0                 # most negative power
discharge_idx = env.n_power_levels - 1  # most positive power

obs, info = env.reset(seed=123)

soc_before = env.soc

action = np.array([discharge_idx, 0])  # dispatch discharge, ignore planning
obs, r, term, trunc, info = env.step(action)

soc_after = env.soc

print("SoC before:", soc_before)
print("SoC after :", soc_after)
print("ΔSoC      :", soc_after - soc_before)
print("Applied MW:", info["P_applied_MW"])

SoC before: 0.5729407452992573
SoC after : 0.3262280117363327
ΔSoC      : -0.2467127335629246
Applied MW: 0.18635


# 3. Sign Consistency: Power and SoC

In [13]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-9

def idx_for_power(env, target_MW: float) -> int:
    return int(np.argmin(np.abs(env.power_levels - target_MW)))

dispatch_charge = idx_for_power(env, -env.P_max_MW)
dispatch_idle   = idx_for_power(env, 0.0)
dispatch_dis    = idx_for_power(env,  env.P_max_MW)

# plan_idx can be anything for this test; keep it simple
plan_idx_const = dispatch_idle

# Build a sequence of 2D actions: [dispatch_idx, plan_idx]
plan = (
    [np.array([dispatch_charge, plan_idx_const], dtype=np.int64)] * 5
    + [np.array([dispatch_idle,   plan_idx_const], dtype=np.int64)] * 3
    + [np.array([dispatch_dis,    plan_idx_const], dtype=np.int64)] * 5
)

obs, info = env.reset()

rows = []
fails = []

for t, a in enumerate(plan):
    soc_before = float(env.soc)

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    P_app = float(info["P_applied_MW"])
    delta = soc_after - soc_before

    ok = True
    # Negative power => charge => SoC should go up (or stay ~same if blocked at max)
    if P_app < -eps and soc_after < soc_before - 1e-8:
        ok = False
    # Positive power => discharge => SoC should go down (or stay ~same if blocked at min)
    if P_app > eps and soc_after > soc_before + 1e-8:
        ok = False
    # ~0 power => SoC should barely move
    if abs(P_app) <= eps and abs(delta) > 1e-6:
        ok = False

    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": float(env.power_levels[dispatch_idx]),
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)
    if not ok:
        fails.append(row)

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num sign fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,0,5,-0.18635,-1.863500e-01,0.635439,0.870213,0.234774,True,-1.023760e+04,2021-06-01 00:00:00,1,False
1,1,0,5,-0.18635,-2.339095e-02,0.870213,0.900000,0.029787,True,-1.232347e+03,2021-06-01 00:30:00,2,False
2,2,0,5,-0.18635,-8.702240e-17,0.900000,0.900000,0.000000,True,-4.584740e-12,2021-06-01 01:00:00,3,False
3,3,0,5,-0.18635,-8.702240e-17,0.900000,0.900000,0.000000,True,-4.428309e-12,2021-06-01 01:30:00,4,False
4,4,0,5,-0.18635,-8.702240e-17,0.900000,0.900000,0.000000,True,-4.545745e-12,2021-06-01 02:00:00,5,False
5,5,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2021-06-01 02:30:00,6,False
6,6,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2021-06-01 03:00:00,7,False
7,7,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2021-06-01 03:30:00,8,False
8,8,10,5,0.18635,1.863500e-01,0.900000,0.655846,-0.244154,True,9.985192e+03,2021-06-01 04:00:00,9,False
9,9,10,5,0.18635,1.863500e-01,0.655846,0.411407,-0.244439,True,1.073964e+04,2021-06-01 04:30:00,10,False


num sign fails: 0


# 4. SOC Protection Function 

In [14]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

eps = 1e-6

# --- SoC grid to test edge + interior ---
soc_test = np.array(
    [
        env.SoC_min,
        env.SoC_min + 1e-3,
        0.2,
        0.5,
        0.9,
        env.SoC_max - 1e-3,
        env.SoC_max,
    ],
    dtype=float
)

# --- Requested power grid ---
power_requested = np.array(
    [
        -env.P_max_MW,
        -0.5 * env.P_max_MW,
        0.0,
        0.5 * env.P_max_MW,
        env.P_max_MW,
    ],
    dtype=float
)

rows = []

for soc in soc_test:
    for P_req in power_requested:

        # --- Call protection ---
        P_app, I_app, V_oc = env._apply_soc_protection(P_req, soc)

        # --- Hard power bounds ---
        ok_power_bounds = abs(P_app) <= env.P_max_MW + eps

        # --- Detect SoC boundary ---
        at_max = soc >= env.SoC_max - 1e-12
        at_min = soc <= env.SoC_min + 1e-12

        # --- Correct blocking behaviour ---
        blocked_charge_at_max = (
            at_max and P_req < 0 and abs(P_app) < eps and abs(I_app) < eps
        )

        blocked_discharge_at_min = (
            at_min and P_req > 0 and abs(P_app) < eps and abs(I_app) < eps
        )

        # --- Compute implied SoC change ---
        if I_app < 0:   # charging
            delta_soc = -(I_app * env.dt_seconds / env.Q_pack_C) * env.eta_ch
        elif I_app > 0: # discharging
            delta_soc = -(I_app * env.dt_seconds / env.Q_pack_C) / env.eta_dis
        else:
            delta_soc = 0.0

        soc_next = soc + delta_soc
        ok_soc_bounds = env.SoC_min - eps <= soc_next <= env.SoC_max + eps

        # --- Power/current consistency (ECM check) ---
        P_recon_W = I_app * (V_oc - I_app * env.R_sys)
        P_recon_MW = P_recon_W / 1e6
        ok_power_consistency = abs(P_recon_MW - P_app) <= 1e-4

        changed = abs(P_app - P_req) > 1e-9

        rows.append({
            "soc": soc,
            "P_req_MW": P_req,
            "P_app_MW": P_app,
            "deltaP": P_app - P_req,
            "changed": changed,
            "I_app_A": I_app,
            "V_oc_V": V_oc,
            "P_reconstructed_MW": P_recon_MW,
            "soc_next": soc_next,
            "ok_power_bounds": ok_power_bounds,
            "ok_soc_bounds": ok_soc_bounds,
            "ok_power_consistency": ok_power_consistency,
            "blocked_charge_at_max": blocked_charge_at_max,
            "blocked_discharge_at_min": blocked_discharge_at_min,
        })

df = pd.DataFrame(rows).sort_values(["soc", "P_req_MW"]).reset_index(drop=True)

# ---- Display full table ----
print("FULL SoC PROTECTION TEST RESULTS")
display(df)

# ---- Failures ----
mask_ok = (
    df["ok_power_bounds"]
    & df["ok_soc_bounds"]
    & df["ok_power_consistency"]
)

fails = df.loc[~mask_ok].reset_index(drop=True)

print(f"num tests: {len(df)} | num fails: {len(fails)}")
display(fails)

# Optional export
df.to_csv("soc_protection_all.csv", index=False)
fails.to_csv("soc_protection_fails.csv", index=False)

FULL SoC PROTECTION TEST RESULTS


,soc,P_req_MW,P_app_MW,deltaP,changed,I_app_A,V_oc_V,P_reconstructed_MW,soc_next,ok_power_bounds,ok_soc_bounds,ok_power_consistency,blocked_charge_at_max,blocked_discharge_at_min
0,0.200,-0.186350,-0.186350,8.326673e-17,False,-135.570757,1352.000000,-0.186350,0.439670,True,True,True,False,False
1,0.200,-0.186350,-0.186350,8.326673e-17,False,-135.570757,1352.000000,-0.186350,0.439670,True,True,True,False,False
2,0.200,-0.093175,-0.093175,4.163336e-17,False,-68.341581,1352.000000,-0.093175,0.320818,True,True,True,False,False
3,0.200,-0.093175,-0.093175,4.163336e-17,False,-68.341581,1352.000000,-0.093175,0.320818,True,True,True,False,False
4,0.200,0.000000,0.000000,0.000000e+00,False,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,False
5,0.200,0.000000,0.000000,0.000000e+00,False,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,False
6,0.200,0.093175,0.000000,-9.317500e-02,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
7,0.200,0.093175,0.000000,-9.317500e-02,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
8,0.200,0.186350,0.000000,-1.863500e-01,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
9,0.200,0.186350,0.000000,-1.863500e-01,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True


num tests: 35 | num fails: 0


,soc,P_req_MW,P_app_MW,deltaP,changed,I_app_A,V_oc_V,P_reconstructed_MW,soc_next,ok_power_bounds,ok_soc_bounds,ok_power_consistency,blocked_charge_at_max,blocked_discharge_at_min


# 5. Day-Ahead (DA) publication & planning logic

In the overlap (realistic) environment, the key market assumption is:

• The agent dispatches in real time every 30 minutes
• The DA prices for tomorrow are not known all the time
• They only become available after the DA publication time (e.g. 13:00)
• Only after that moment should the agent:
	•	see the tomorrow DA curve in the observation
	•	be able to write meaningful values into tomorrow_plan

In [15]:
import numpy as np
import pandas as pd

def run_da_publication_sanity(env, seed=123, n_steps=80):
    obs, reset_info = env.reset(seed=seed)
    rows = []

    for k in range(n_steps):
        obs_before = obs.copy()  # s_t

        action = env.action_space.sample()
        obs, r, term, trunc, info = env.step(action)  # obs is now s_{t+1}

        # DA curve visibility should be evaluated from s_t (obs_before)
        da_curve = obs_before[6:]
        curve_nonzero = bool(np.any(np.abs(da_curve) > 1e-6))

        rows.append({
            "k": k,
            "trade_ts": info["trade_ts"],        # current step time (trade clock)
            "tau": info["tau"],
            "da_available": info["da_available"],
            "tomorrow_curve_nonzero": curve_nonzero,
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    df["trade_ts"] = pd.to_datetime(df["trade_ts"])
    df["dt_min"] = df["trade_ts"].diff().dt.total_seconds() / 60.0

    # Expected publication time for the starting trade day
    trade_day0 = df["trade_ts"].iloc[0].floor("D")
    df["publish_ts_expected"] = trade_day0 + pd.Timedelta(hours=env.publish_hour)

    # Helpful: where the gate first flips
    df["gate_mismatch"] = df["da_available"] != df["tomorrow_curve_nonzero"]

    return df

df_da = run_da_publication_sanity(env, seed=123, n_steps=80)
df_da

,k,trade_ts,tau,da_available,tomorrow_curve_nonzero,dt_min,publish_ts_expected,gate_mismatch
0,0,2022-02-19 00:00:00,1,False,False,NaN,2022-02-19 13:00:00,False
1,1,2022-02-19 00:30:00,2,False,False,30.0,2022-02-19 13:00:00,False
2,2,2022-02-19 01:00:00,3,False,False,30.0,2022-02-19 13:00:00,False
3,3,2022-02-19 01:30:00,4,False,False,30.0,2022-02-19 13:00:00,False
4,4,2022-02-19 02:00:00,5,False,False,30.0,2022-02-19 13:00:00,False
...,...,...,...,...,...,...,...,...
75,75,2022-02-20 13:30:00,28,True,True,30.0,2022-02-19 13:00:00,False
76,76,2022-02-20 14:00:00,29,True,True,30.0,2022-02-19 13:00:00,False
77,77,2022-02-20 14:30:00,30,True,True,30.0,2022-02-19 13:00:00,False
78,78,2022-02-20 15:00:00,31,True,True,30.0,2022-02-19 13:00:00,False


# 6. Step Level Invariants
## 6.1. SoC always within bounds

In [16]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

N_steps = 20_000  # set to len(data) if you want a full pass

obs, info = env.reset()
fails = []

for t in range(N_steps):
    a = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(a)

    soc = float(obs[0])
    if not (env.SoC_min - eps <= soc <= env.SoC_max + eps):
        fails.append({
            "t": t,
            "action": int(a),
            "soc": soc,
            "SoC_min": env.SoC_min,
            "SoC_max": env.SoC_max,
            "P_requested_MW": float(env.power_levels[a]),
            "P_applied_MW": float(info.get("P_applied_MW", np.nan)),
            "reward": float(reward),
        })
        break  # stop at first failure for fast debugging

    if terminated or truncated:
        break

print(f"ran steps: {t+1}")
print(f"num fails: {len(fails)}")
if fails:
    display(pd.DataFrame(fails))

ran steps: 1440
num fails: 0


## 6.2. Energy Accounting Check

In [17]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-9

obs, info = env.reset()
rows = []
fails = []

N = 30

for t in range(N):
    a = env.action_space.sample()
    P_req = float(env.power_levels[dispatch_idx])
    
    soc_before = env.soc
    obs, reward, terminated, truncated, info = env.step(a)
    soc_after = env.soc

    P_app = float(info["P_applied_MW"])
    E_MWh = P_app * env.dt_hours

    ok_mag = (abs(E_MWh) <= env.P_max_MW * env.dt_hours + 1e-6)

    ok_sign = ((E_MWh >= -eps) if (P_app >= 0) else (E_MWh <= eps)) 

    dispatch_idx = int(a[0])
    plan_idx = int(a[1])
    
    row = {
        "t": t,
        "dispatch_action": dispatch_idx,
        "plan_idx":plan_idx,
        "P_req_MW": P_req,
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "dt_hours": float(env.dt_hours),
        "E_MWh" : E_MWh,
        "ok mag": ok_mag,
        "ok_sign": ok,
    }

    rows.append(row)
    if not ok:
        fails.append(row)
    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

    

,t,dispatch_action,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,dt_hours,E_MWh,ok mag,ok_sign
0,0,7,7,0.18635,0.074540,0.499354,0.401446,0.0,0.5,0.037270,True,True
1,1,9,0,0.07454,0.149080,0.401446,0.204168,0.0,0.5,0.074540,True,True
2,2,1,6,0.14908,-0.149080,0.204168,0.396459,0.0,0.5,-0.074540,True,True
3,3,8,4,-0.14908,0.111810,0.396459,0.249007,0.0,0.5,0.055905,True,True
4,4,4,0,0.11181,-0.037270,0.249007,0.297374,0.0,0.5,-0.018635,True,True
5,5,6,10,-0.03727,0.037270,0.297374,0.248464,0.0,0.5,0.018635,True,True
6,6,4,4,0.03727,-0.037270,0.248464,0.296833,0.0,0.5,-0.018635,True,True
7,7,6,0,-0.03727,0.037270,0.296833,0.247921,0.0,0.5,0.018635,True,True
8,8,2,0,0.03727,-0.111810,0.247921,0.392080,0.0,0.5,-0.055905,True,True
9,9,0,3,-0.11181,-0.186350,0.392080,0.629068,0.0,0.5,-0.093175,True,True


# 7. Reward Decomposition Checks

## 6.1. Profit term sanity with constant price

In [18]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

# Make carbon term vanish
env.lambda_ci = 0.0

obs, info = env.reset(seed=123)

rows = []
fails = []

N = 50

for t in range(N):
    a = env.action_space.sample()
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    soc_before = float(env.soc)
    obs, reward, terminated, truncated, info = env.step(a)
    soc_after = float(env.soc)

    P_app = float(info["P_applied_MW"])
    price_now = float(info["price_now"])
    dt = float(env.dt_hours)

    # Profit definition used in env: profit = E_MWh * price_now
    E_MWh = P_app * dt
    profit_calc = E_MWh * price_now

    ok_profit_only = np.isclose(float(reward), profit_calc, atol=1e-5)

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_app_MW": P_app,
        "price_now": price_now,
        "E_MWh": E_MWh,
        "reward_env": float(reward),
        "profit_calc": profit_calc,
        "ok_profit_only": ok_profit_only,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "tau": info.get("tau", None),
        "trade_ts": info.get("trade_ts", None),
    }
    rows.append(row)

    if not ok_profit_only:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_app_MW,price_now,E_MWh,reward_env,profit_calc,ok_profit_only,soc_before,soc_after,tau,trade_ts
0,0,1,5,-1.490800e-01,100.000000,-7.454000e-02,-7.454000e+00,-7.454000e+00,True,0.572941,0.762595,1,2022-02-19 00:00:00
1,1,6,3,3.727000e-02,112.000000,1.863500e-02,2.087120e+00,2.087120e+00,True,0.762595,0.714384,2,2022-02-19 00:30:00
2,2,4,4,-3.727000e-02,140.000000,-1.863500e-02,-2.608900e+00,-2.608900e+00,True,0.714384,0.761808,3,2022-02-19 01:00:00
3,3,2,6,-1.092778e-01,80.000000,-5.463891e-02,-4.371113e+00,-4.371113e+00,True,0.761808,0.900000,4,2022-02-19 01:30:00
4,4,3,0,-8.702240e-17,97.599998,-4.351120e-17,-4.246693e-15,-4.246693e-15,True,0.900000,0.900000,5,2022-02-19 02:00:00
5,5,1,6,-8.702240e-17,70.000000,-4.351120e-17,-3.045784e-15,-3.045784e-15,True,0.900000,0.900000,6,2022-02-19 02:30:00
6,6,3,8,-8.702240e-17,88.400002,-4.351120e-17,-3.846390e-15,-3.846390e-15,True,0.900000,0.900000,7,2022-02-19 03:00:00
7,7,1,5,-8.702240e-17,53.000000,-4.351120e-17,-2.306093e-15,-2.306093e-15,True,0.900000,0.900000,8,2022-02-19 03:30:00
8,8,9,8,1.490800e-01,70.000000,7.454000e-02,5.217800e+00,5.217800e+00,True,0.900000,0.705336,9,2022-02-19 04:00:00
9,9,1,1,-1.490800e-01,50.000000,-7.454000e-02,-3.727000e+00,-3.727000e+00,True,0.705336,0.893245,10,2022-02-19 04:30:00


num fails: 0


## 7.2. Reward Check including Carbon 

In [19]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

env.lambda_ci = float(env.lambda_ci)  # keep as configured

obs, info = env.reset(seed=123)

rows = []
fails = []

N = 50

for t in range(N):
    a = env.action_space.sample()
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    obs, reward, terminated, truncated, info = env.step(a)

    P_app = float(info["P_applied_MW"])
    price_now = float(info["price_now"])
    ci_now = float(info["ci_now"])
    dt = float(env.dt_hours)

    E_MWh = P_app * dt
    profit_calc = E_MWh * price_now

    E_imp_kWh = max(-E_MWh, 0.0) * 1000.0
    E_exp_kWh = max(E_MWh, 0.0) * 1000.0
    carbon_penalty_calc = env.lambda_ci * (E_imp_kWh - E_exp_kWh) * ci_now

    reward_calc = profit_calc - carbon_penalty_calc

    ok_reward = np.isclose(float(reward), reward_calc, atol=1e-5)

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_app_MW": P_app,
        "price_now": price_now,
        "ci_now": ci_now,
        "E_MWh": E_MWh,
        "profit_calc": profit_calc,
        "carbon_penalty_calc": carbon_penalty_calc,
        "reward_env": float(reward),
        "reward_calc": reward_calc,
        "ok_reward": ok_reward,
        "tau": info.get("tau", None),
        "trade_ts": info.get("trade_ts", None),
    }
    rows.append(row)

    if not ok_reward:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_app_MW,price_now,ci_now,E_MWh,profit_calc,carbon_penalty_calc,reward_env,reward_calc,ok_reward,tau,trade_ts
0,0,0,0,-1.863500e-01,100.000000,48.0,-9.317500e-02,-9.317500e+00,4.025160e+03,-4.034477e+03,-4.034477e+03,True,1,2022-02-19 00:00:00
1,1,5,0,0.000000e+00,112.000000,49.0,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,True,2,2022-02-19 00:30:00
2,2,7,0,7.454000e-02,140.000000,54.0,3.727000e-02,5.217800e+00,-1.811322e+03,1.816540e+03,1.816540e+03,True,3,2022-02-19 01:00:00
3,3,9,10,1.490800e-01,80.000000,53.0,7.454000e-02,5.963200e+00,-3.555558e+03,3.561521e+03,3.561521e+03,True,4,2022-02-19 01:30:00
4,4,9,6,1.490800e-01,97.599998,54.0,7.454000e-02,7.275104e+00,-3.622644e+03,3.629919e+03,3.629919e+03,True,5,2022-02-19 02:00:00
5,5,8,8,9.158746e-02,70.000000,56.0,4.579373e-02,3.205561e+00,-2.308004e+03,2.311210e+03,2.311210e+03,True,6,2022-02-19 02:30:00
6,6,4,7,-3.727000e-02,88.400002,57.0,-1.863500e-02,-1.647334e+00,9.559755e+02,-9.576228e+02,-9.576228e+02,True,7,2022-02-19 03:00:00
7,7,2,7,-1.118100e-01,53.000000,57.0,-5.590500e-02,-2.962965e+00,2.867926e+03,-2.870889e+03,-2.870889e+03,True,8,2022-02-19 03:30:00
8,8,4,4,-3.727000e-02,70.000000,58.0,-1.863500e-02,-1.304450e+00,9.727470e+02,-9.740514e+02,-9.740514e+02,True,9,2022-02-19 04:00:00
9,9,9,10,1.490800e-01,50.000000,57.0,7.454000e-02,3.727000e+00,-3.823902e+03,3.827629e+03,3.827629e+03,True,10,2022-02-19 04:30:00


num fails: 0


# 8. Plan Rollover
## 8.1. Yesterday plan becomes today_plan at rollover

In [20]:
env = BatteryEnv(env_config)
obs, info = env.reset(seed=123)

pattern = np.arange(48) % env.n_power_levels
env.tomorrow_plan[:] = pattern.copy()

# Freeze DA availability (so step() cannot overwrite tomorrow_plan)
env.da_publish_ts[:] = env.trade_ts + np.timedelta64(3650, "D")  # +10 years

# Fast-forward to rollover
steps_left = 48 - env.slot0
for _ in range(steps_left):
    obs, r, term, trunc, inf = env.step(env.action_space.sample())

ok = np.all(env.today_plan == pattern)
print("today_plan matches yesterday's tomorrow_plan:", ok)
if not ok:
    print("max abs diff:", np.max(np.abs(env.today_plan - pattern)))

today_plan matches yesterday's tomorrow_plan: True


In [21]:
env = BatteryEnv(env_config, publish_hour=0)  # DA available from 00:00
obs, info = env.reset(seed=123)

pattern = np.arange(48) % env.n_power_levels

for _ in range(48):
    tau0 = env.slot0              # 0..47
    plan_idx = int(pattern[tau0])
    dispatch_idx = 0
    action = np.array([dispatch_idx, plan_idx], dtype=int)
    obs, r, term, trunc, inf = env.step(action)

ok = np.all(env.today_plan == pattern)
print("today_plan matches yesterday's tomorrow_plan:", ok)
if not ok:
    print("max abs diff:", np.max(np.abs(env.today_plan - pattern)))
    print("today_plan head:", env.today_plan[:10])
    print("pattern head:", pattern[:10])

today_plan matches yesterday's tomorrow_plan: True


In [22]:
obs, info = env.reset(seed=123)
print("First step da_available:", env._da_available_now(int(env.current_day_idxs[env.slot0])))
print("First trade_ts:", env.trade_ts[int(env.current_day_idxs[env.slot0])])
print("First da_publish_ts:", env.da_publish_ts[int(env.current_day_idxs[env.slot0])])

First step da_available: True
First trade_ts: 2022-02-19T00:00:00.000000000
First da_publish_ts: 2022-02-19T00:00:00.000000000


In [23]:
obs, info = env.reset(seed=123)
flags = []
for _ in range(48):
    idx = int(env.current_day_idxs[env.slot0])
    flags.append(env._da_available_now(idx))
    obs, r, term, trunc, inf = env.step(env.action_space.sample())
print("Any DA available today?", any(flags))
print("Count true:", sum(flags))

Any DA available today? True
Count true: 48


In [24]:
env2 = BatteryEnv(env_config, publish_hour=13)
obs, info = env2.reset(seed=123)

idx0 = int(env2.current_day_idxs[env2.slot0])
print("trade_ts:", env2.trade_ts[idx0])
print("da_publish_ts:", env2.da_publish_ts[idx0])
print("da_available:", env2._da_available_now(idx0))

trade_ts: 2022-02-19T00:00:00.000000000
da_publish_ts: 2022-02-19T13:00:00.000000000
da_available: False


## 8.1. Full Test of Plan Rollover 

In [6]:
import numpy as np
import pandas as pd

def action_for_power_idx(env, target_MW: float) -> int:
    return int(np.argmin(np.abs(env.power_levels - target_MW)))

def run_designB_tests(env, seed=123, publish_hour=13):
    env.publish_hour = publish_hour

    obs, info = env.reset(seed=seed)
    nA = env.n_power_levels

    def cur_idx():
        return int(env.current_day_idxs[env.slot0])

    def cur_tau0(idx):
        return int(env.tau[idx]) - 1  # 0..47

    # ----------------------------
    # TEST 1: tomorrow_plan gated by DA availability
    # ----------------------------
    idle = action_for_power_idx(env, 0.0)
    pattern = np.arange(48) % nA

    before_changes = 0
    after_changes = 0

    for _ in range(48):
        idx = cur_idx()
        tau0 = cur_tau0(idx)

        da_avail = env._da_available_now(idx)

        old = int(env.tomorrow_plan[tau0])
        a = np.array([idle, int(pattern[tau0])], dtype=int)
        obs, r, term, trunc, inf = env.step(a)
        new = int(env.tomorrow_plan[tau0])

        if da_avail:
            after_changes += (new != old)
        else:
            before_changes += (new != old)

        if term or trunc:
            break

    print("TEST 1 (DA gate)")
    print("  any writes before publish? ", before_changes > 0)
    print("  any writes after publish?  ", after_changes > 0)

    # ----------------------------
    # TEST 2: rollover correctness
    # Only slots that were actually written (>=0) should match pattern.
    # ----------------------------
    written_mask = env.today_plan >= 0

    if np.any(written_mask):
        today_ok = np.all(env.today_plan[written_mask] == pattern[written_mask])
        max_diff = int(np.max(np.abs(env.today_plan[written_mask] - pattern[written_mask])))
    else:
        today_ok = True
        max_diff = 0

    print("\nTEST 2 (rollover plan)")
    print("  today_plan matches pattern on written slots:", today_ok)
    print("  max abs diff (written slots only):", max_diff)
    print("  today_plan head:", env.today_plan[:10])
    print("  pattern head:   ", pattern[:10])
    print("  num planned slots:", int(np.sum(written_mask)), "out of 48")

    # ----------------------------
    # TEST 3: execution uses today_plan when it exists
    # If today_plan[tau] == -1, agent dispatch is allowed → auto-pass those rows.
    # When today_plan[tau] >= 0, compare to *protected* applied power.
    # ----------------------------
    rows = []
    fails = 0

    for _ in range(48):
        idx = cur_idx()
        tau0 = cur_tau0(idx)

        a = env.action_space.sample()
        dispatch_agent = int(a[0])
        plan_agent = int(a[1])

        soc_before = float(env.soc)

        planned_idx = int(env.today_plan[tau0])
        dispatch_eff_expected = planned_idx if planned_idx >= 0 else dispatch_agent

        P_req_expected = float(env.power_levels[dispatch_eff_expected])
        P_app_expected, _, _ = env._apply_soc_protection(P_req_expected, soc_before)

        obs, r, term, trunc, inf = env.step(a)

        got_P = float(inf["P_applied_MW"])
        ok = np.isclose(got_P, P_app_expected, atol=1e-6)

        rows.append({
            "trade_ts": inf["trade_ts"],
            "tau": int(tau0 + 1),
            "planned_idx_today": planned_idx,
            "dispatch_idx_agent": dispatch_agent,
            "dispatch_idx_eff_expected": dispatch_eff_expected,
            "dispatch_idx_exec_info": inf["dispatch_idx_exec"],
            "P_req_expected": P_req_expected,
            "P_app_expected": P_app_expected,
            "got_P": got_P,
            "ok": ok,
        })

        fails += (not ok)

        if term or trunc:
            break

    df = pd.DataFrame(rows)

    print("\nTEST 3 (execution uses today_plan when planned)")
    print("  fails:", int(fails), "out of", len(df))

    return df

# usage:
df_exec = run_designB_tests(env, seed=123, publish_hour=13)
display(df_exec.head(20))
display(df_exec[df_exec["ok"] == False].head(50))

TEST 1 (DA gate)
  any writes before publish?  False
  any writes after publish?   True

TEST 2 (rollover plan)
  today_plan matches pattern on written slots: True
  max abs diff (written slots only): 0
  today_plan head: [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
  pattern head:    [0 1 2 3 4 5 6 7 8 9]
  num planned slots: 22 out of 48

TEST 3 (execution uses today_plan when planned)
  fails: 0 out of 48


,trade_ts,tau,planned_idx_today,dispatch_idx_agent,dispatch_idx_eff_expected,dispatch_idx_exec_info,P_req_expected,P_app_expected,got_P,ok
0,2021-01-13 00:00:00,1,-1,7,7,7,0.07454,7.454000e-02,7.454000e-02,True
1,2021-01-13 00:30:00,2,-1,4,4,4,-0.03727,-3.727000e-02,-3.727000e-02,True
2,2021-01-13 01:00:00,3,-1,7,7,7,0.07454,7.454000e-02,7.454000e-02,True
3,2021-01-13 01:30:00,4,-1,1,1,1,-0.14908,-1.490800e-01,-1.490800e-01,True
4,2021-01-13 02:00:00,5,-1,1,1,1,-0.14908,-1.490800e-01,-1.490800e-01,True
5,2021-01-13 02:30:00,6,-1,3,3,3,-0.07454,-7.454000e-02,-7.454000e-02,True
6,2021-01-13 03:00:00,7,-1,3,3,3,-0.07454,-5.761042e-02,-5.761042e-02,True
7,2021-01-13 03:30:00,8,-1,4,4,4,-0.03727,-1.740448e-16,-1.740448e-16,True
8,2021-01-13 04:00:00,9,-1,1,1,1,-0.14908,-1.740448e-16,-1.740448e-16,True
9,2021-01-13 04:30:00,10,-1,9,9,9,0.14908,1.490800e-01,1.490800e-01,True


,trade_ts,tau,planned_idx_today,dispatch_idx_agent,dispatch_idx_eff_expected,dispatch_idx_exec_info,P_req_expected,P_app_expected,got_P,ok


# 9. Observation Correctness
## 9.1 Obs matches internal state at current step

In [9]:
import numpy as np
import pandas as pd

def run_obs_checks(env, seed=123, n_steps=4000, eps=1e-6, verbose=True):
    obs, info = env.reset(seed=seed)

    fails = []

    def cur_idx():
        return int(env.current_day_idxs[env.slot0])

    def record_fail(k, kind, details):
        row = {"k": k, "kind": kind}
        row.update(details)
        fails.append(row)

    for k in range(n_steps):
        idx = cur_idx()

        # ===== expected internal values at *current* slot =====
        soc_exp = float(env.soc)
        spot_exp = float(env.spot_price[idx])
        ci_exp = float(env.ci[idx])
        tau_exp = float(env.tau[idx])             # 1..48
        pprev_exp = float(env.p_prev)
        da_avail_exp = 1.0 if env._da_available_now(idx) else 0.0

        # expected tomorrow curve (per your env definition)
        curve_exp = env._get_tomorrow_da_curve().astype(np.float32) if da_avail_exp > 0.5 else np.zeros(48, dtype=np.float32)

        # ===== B1: check main obs fields =====
        if not np.isclose(float(obs[0]), soc_exp, atol=eps):
            record_fail(k, "B1_soc", {"obs_soc": float(obs[0]), "exp_soc": soc_exp})

        if not np.isclose(float(obs[1]), spot_exp, atol=eps):
            record_fail(k, "B1_spot", {"obs_spot": float(obs[1]), "exp_spot": spot_exp})

        if not np.isclose(float(obs[2]), ci_exp, atol=eps):
            record_fail(k, "B1_ci", {"obs_ci": float(obs[2]), "exp_ci": ci_exp})

        if not np.isclose(float(obs[3]), tau_exp, atol=eps):
            record_fail(k, "B1_tau", {"obs_tau": float(obs[3]), "exp_tau": tau_exp})

        if not np.isclose(float(obs[4]), pprev_exp, atol=eps):
            record_fail(k, "B1_p_prev", {"obs_p_prev": float(obs[4]), "exp_p_prev": pprev_exp})

        if not np.isclose(float(obs[5]), da_avail_exp, atol=eps):
            record_fail(k, "B1_da_flag", {"obs_da": float(obs[5]), "exp_da": da_avail_exp})

        # ===== B2: tomorrow curve logic =====
        curve_obs = obs[6:].astype(np.float32)
        if da_avail_exp < 0.5:
            # must be (near) all zeros
            if np.any(np.abs(curve_obs) > 1e-6):
                record_fail(k, "B2_curve_should_be_zero", {
                    "max_abs_curve": float(np.max(np.abs(curve_obs))),
                    "trade_ts": str(pd.Timestamp(env.trade_ts[idx])),
                    "da_publish_ts": str(pd.Timestamp(env.da_publish_ts[idx])),
                })
        else:
            # must match today's 48 DA values
            if not np.allclose(curve_obs, curve_exp, atol=1e-6):
                record_fail(k, "B2_curve_mismatch", {
                    "max_abs_diff": float(np.max(np.abs(curve_obs - curve_exp))),
                    "trade_ts": str(pd.Timestamp(env.trade_ts[idx])),
                    "da_publish_ts": str(pd.Timestamp(env.da_publish_ts[idx])),
                })

        # ===== B3: obs within Box bounds =====
        low = env.observation_space.low
        high = env.observation_space.high
        if np.any(obs < low - 1e-5) or np.any(obs > high + 1e-5):
            # store worst offending dimension
            below = obs - low
            above = obs - high
            worst_below_i = int(np.argmin(below))
            worst_above_i = int(np.argmax(above))
            record_fail(k, "B3_obs_out_of_bounds", {
                "worst_below_i": worst_below_i,
                "obs_val_below": float(obs[worst_below_i]),
                "low_val": float(low[worst_below_i]),
                "worst_above_i": worst_above_i,
                "obs_val_above": float(obs[worst_above_i]),
                "high_val": float(high[worst_above_i]),
                "trade_ts": str(pd.Timestamp(env.trade_ts[idx])),
            })

        # advance with random action
        a = env.action_space.sample()
        obs, r, term, trunc, inf = env.step(a)
        if term or trunc:
            break

    df_fails = pd.DataFrame(fails)
    if verbose:
        print(f"ran steps: {k+1}")
        print(f"num fails: {len(df_fails)}")
        if len(df_fails):
            display(df_fails.head(50))

    return df_fails

# Usage:
df_fails = run_obs_checks(env, seed=123, n_steps=4000)
display(df_fails)

ran steps: 336
num fails: 0


""


# 10 Episode / termination / indexing

#### 10.1. Step to end of episode_days → terminated True exactly when expected
#### 10.2. Dataset end → truncated True (and no crash)
#### 10.3. Reset reproducibility (seeded)
#### same seed → same start_day and same initial SoC (given your RNG usage)

In [11]:
import numpy as np
import pandas as pd

def run_episode_boundary_checks(env, seed=123, episode_days_test=3):
    # Save / restore
    orig_episode_days = int(env.episode_days)
    orig_randomize = bool(env.randomize_init_soc)

    try:
        env.episode_days = int(episode_days_test)
        env.randomize_init_soc = True

        obs, info = env.reset(seed=seed)
        start_day_1 = info["start_day"]
        init_soc_1 = info["init_soc"]

        # ---- G1: step until done, count steps ----
        steps = 0
        terminated = truncated = False

        while True:
            a = env.action_space.sample()
            obs, r, term, trunc, inf = env.step(a)
            steps += 1
            if term or trunc:
                terminated, truncated = term, trunc
                break

        expected_steps = env.episode_days * 48
        print("G1) episode length")
        print("  steps:", steps)
        print("  expected:", expected_steps)
        print("  terminated:", terminated, "| truncated:", truncated)

        # Termination should be True when you hit episode_days
        # (Truncation can happen if dataset ends early, but with valid_days this is usually not the case.)
        ok_len = (steps == expected_steps) and terminated
        print("  ok_len:", ok_len)

        # ---- G2: reset reproducibility ----
        obs2, info2 = env.reset(seed=seed)
        start_day_2 = info2["start_day"]
        init_soc_2 = info2["init_soc"]

        print("\nG2) reset reproducibility (same seed)")
        print("  start_day same:", start_day_1 == start_day_2)
        print("  init_soc same:", np.isclose(init_soc_1, init_soc_2, atol=1e-12))
        print("  start_day:", start_day_1)
        print("  init_soc:", init_soc_1)

        return {
            "steps": steps,
            "expected_steps": expected_steps,
            "terminated": terminated,
            "truncated": truncated,
            "ok_len": ok_len,
            "start_day_same": (start_day_1 == start_day_2),
            "init_soc_same": bool(np.isclose(init_soc_1, init_soc_2, atol=1e-12)),
        }

    finally:
        env.episode_days = orig_episode_days
        env.randomize_init_soc = orig_randomize


# Usage:
out = run_episode_boundary_checks(env, seed=123, episode_days_test=3)
print(out)

G1) episode length
  steps: 144
  expected: 144
  terminated: True | truncated: False
  ok_len: True

G2) reset reproducibility (same seed)
  start_day same: True
  init_soc same: True
  start_day: 2022-03-07T00:00:00.000000000
  init_soc: 0.5729407452992573
{'steps': 144, 'expected_steps': 144, 'terminated': True, 'truncated': False, 'ok_len': True, 'start_day_same': True, 'init_soc_same': True}


# G) “No cheating” checks

#### G1. Before publish, tomorrow_curve must be exactly zeros
#### G2. After publish, tomorrow_curve should be visible, but only for the correct day (no leaking “future of future”)
#### G3. If you want “reactive dispatch only” mode: confirm agent dispatch can still act freely when no plan (planned_idx=-1)

In [12]:
import numpy as np
import pandas as pd

def run_no_cheating_DA_checks(env, seed=123, eps=1e-6):
    obs, info = env.reset(seed=seed)

    # Force no plan interference
    env.today_plan[:] = -1
    env.tomorrow_plan[:] = -1

    rows = []
    fails = []

    # Snapshot the "true" DA curve for the current trade day (48 rows)
    idxs_day = env.current_day_idxs
    true_curve = env.da_price[idxs_day].astype(np.float32)

    for k in range(48):  # one trade day
        idx = int(env.current_day_idxs[env.slot0])
        da_avail = env._da_available_now(idx)

        curve = obs[6:].astype(np.float32)
        is_zero = bool(np.all(np.abs(curve) <= eps))
        matches_true = bool(np.all(np.abs(curve - true_curve) <= eps))

        # --- G1/G2 conditions ---
        ok = True
        if not da_avail:
            # G1: before publish MUST be all zeros
            if not is_zero:
                ok = False
        else:
            # G2: after publish MUST equal the trade-day DA curve
            if not matches_true:
                ok = False

        row = {
            "k": k,
            "trade_ts": env.trade_ts[idx],
            "tau": int(env.tau[idx]),
            "da_available": bool(da_avail),
            "curve_is_zero": is_zero,
            "curve_matches_true_day_curve": matches_true,
            "ok": ok,
        }
        rows.append(row)
        if not ok:
            fails.append(row)

        # step
        a = env.action_space.sample()
        obs, r, term, trunc, inf = env.step(a)
        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("G1/G2 | fails:", len(fails))
    if fails:
        display(pd.DataFrame(fails))

    # Extra: check that the visible curve does NOT change within the same trade day after publish
    # (if it changes, you might be leaking something time-varying that shouldn't change)
    df_after = df[df["da_available"] == True].copy()
    if len(df_after) > 1:
        # Recompute observed curves for those steps by replaying quickly would be heavy,
        # so we do a lighter proxy check: once da_available becomes true, it should stay true for the rest of day.
        da_monotone = bool(np.all(np.diff(df["da_available"].astype(int)) >= 0))
        print("  DA availability monotone (False->True only once):", da_monotone)

    return df

# Usage:
df_da_nc = run_no_cheating_DA_checks(env, seed=123)
display(df_da_nc)

G1/G2 | fails: 0
  DA availability monotone (False->True only once): True


,k,trade_ts,tau,da_available,curve_is_zero,curve_matches_true_day_curve,ok
0,0,2021-01-12 00:00:00,1,False,True,False,True
1,1,2021-01-12 00:30:00,2,False,True,False,True
2,2,2021-01-12 01:00:00,3,False,True,False,True
3,3,2021-01-12 01:30:00,4,False,True,False,True
4,4,2021-01-12 02:00:00,5,False,True,False,True
5,5,2021-01-12 02:30:00,6,False,True,False,True
6,6,2021-01-12 03:00:00,7,False,True,False,True
7,7,2021-01-12 03:30:00,8,False,True,False,True
8,8,2021-01-12 04:00:00,9,False,True,False,True
9,9,2021-01-12 04:30:00,10,False,True,False,True


In [13]:
import numpy as np
import pandas as pd

def run_no_future_of_future_check(env, seed=123, eps=1e-6):
    obs, info = env.reset(seed=seed)
    env.today_plan[:] = -1
    env.tomorrow_plan[:] = -1

    rows, fails = [], []

    for k in range(48):
        idx = int(env.current_day_idxs[env.slot0])
        da_avail = env._da_available_now(idx)

        # "truth" for THIS trade day at THIS step
        true_curve_now = env.da_price[env.current_day_idxs].astype(np.float32)
        curve = obs[6:].astype(np.float32)

        if da_avail:
            ok = bool(np.all(np.abs(curve - true_curve_now) <= eps))
        else:
            ok = bool(np.all(np.abs(curve) <= eps))

        rows.append({
            "k": k,
            "trade_ts": env.trade_ts[idx],
            "trade_day": str(env.current_day),
            "tau": int(env.tau[idx]),
            "da_available": bool(da_avail),
            "ok_curve_day_alignment": ok,
        })

        if not ok:
            fails.append(rows[-1])

        a = env.action_space.sample()
        obs, r, term, trunc, inf = env.step(a)
        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("G2 (no future-of-future) | fails:", len(fails))
    if fails:
        display(pd.DataFrame(fails))
    return df

# Usage:
df_future = run_no_future_of_future_check(env, seed=123)

G2 (no future-of-future) | fails: 0


This checks that when today_plan[tau0] == -1, the executed dispatch index equals the agent’s dispatch index, and the applied power equals protection on that request (given soc_before).

In [18]:
import numpy as np
import pandas as pd

def run_reactive_dispatch_freedom_check(env, seed=123, n_steps=200, eps=1e-6):
    obs, info = env.reset(seed=seed)

    # Force "no plan" mode for entire run
    env.today_plan[:] = -1
    env.tomorrow_plan[:] = -1

    rows, fails = [], []

    for t in range(n_steps):
        idx = int(env.current_day_idxs[env.slot0])
        tau0 = int(env.tau[idx]) - 1

        # ensure it's actually "no plan" at this slot
        assert int(env.today_plan[tau0]) == -1

        a = env.action_space.sample()
        dispatch_agent = int(a[0])
        plan_agent = int(a[1])

        soc_before = float(env.soc)
        P_req = float(env.power_levels[dispatch_agent])
        P_app_expected, _, _ = env._apply_soc_protection(P_req, soc_before)

        obs, r, term, trunc, inf = env.step(a)

        # what env executed
        exec_idx = int(inf["dispatch_idx_exec"])
        got_P = float(inf["P_applied_MW"])

        ok_idx = (exec_idx == dispatch_agent)
        ok_P = np.isclose(got_P, P_app_expected, atol=1e-6)

        ok = bool(ok_idx and ok_P)

        row = {
            "t": t,
            "trade_ts": inf.get("trade_ts", None),
            "tau": int(inf.get("tau", -1)),
            "planned_idx_today": int(inf["planned_idx_today"]),
            "dispatch_idx_agent": dispatch_agent,
            "dispatch_idx_exec": exec_idx,
            "P_req_MW": P_req,
            "P_app_expected": float(P_app_expected),
            "P_app_got": got_P,
            "ok_idx": ok_idx,
            "ok_P": ok_P,
            "ok": ok,
        }
        rows.append(row)
        if not ok:
            fails.append(row)
            # don't break immediately; useful to see pattern
        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("G3 (reactive dispatch freedom when no plan) | fails:", len(fails))
    if fails:
        display(pd.DataFrame(fails).head(20))
    return df

# Usage:
df_free = run_reactive_dispatch_freedom_check(env, seed=123, n_steps=200)

AssertionError: 